# AI-Based Network Anomaly Detection Using Zeek Connection Logs

End-to-end pipeline: Zeek `conn.log` parsing → preprocessing & feature engineering → unsupervised anomaly detection (Isolation Forest, One-Class SVM, Local Outlier Factor, optional Autoencoder) → export for Elasticsearch/Kibana visualization.

Originally developed and run in Google Colab.

In [ ]:
# ============================================================================
# CELL 1: SETUP & DATA LOADING
# ============================================================================
# CRITICAL: Run this ENTIRE cell - do not skip the imports at the top
# ============================================================================

# IMPORTS FIRST - DO NOT SKIP THESE LINES
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA

from datetime import datetime

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)

print("✓ All libraries imported")
print(f"Timestamp: {datetime.now()}")

In [ ]:
# ============================================================================
# DATA LOADING FUNCTION
# ============================================================================
def parse_zeek_log(filepath):
    """Parse Zeek/Bro conn.log files."""
    separator = '\t'
    columns = []

    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('#separator'):
                separator = '\t' if '\\x09' in line else ' '
            elif line.startswith('#fields'):
                columns = line.strip().split('\t')[1:]
            elif not line.startswith('#'):
                break

    print(f"Detected {len(columns)} columns")

    df = pd.read_csv(
        filepath,
        sep=separator,
        names=columns,
        comment='#',
        skip_blank_lines=True,
        low_memory=False,
        on_bad_lines='skip'
    )
    return df

In [ ]:
# ============================================================================
# UPLOAD FILE
# ============================================================================
from google.colab import files
print("\n" + "="*80)
print("Please upload your conn.log file:")
print("="*80)
uploaded = files.upload()
filename = list(uploaded.keys())[0]

In [ ]:
# ============================================================================
# PARSE THE FILE
# ============================================================================
print(f"\nParsing: {filename}")
df_conn = parse_zeek_log(filename)

print(f"\n✓ Loaded: {df_conn.shape[0]:,} rows × {df_conn.shape[1]} columns")
print(f"  Memory: {df_conn.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# ============================================================================
# HANDLE ZEEK MISSING VALUES
# ============================================================================
df_conn.replace(['-', '(empty)', 'empty'], np.nan, inplace=True)
print("✓ Zeek missing value indicators replaced with NaN")

In [ ]:
# ============================================================================
# DATA TYPE CONVERSION
# ============================================================================
print("\nConverting data types...")

numeric_candidates = [
    'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes',
    'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes'
]

for col in numeric_candidates:
    if col in df_conn.columns:
        df_conn[col] = pd.to_numeric(df_conn[col], errors='coerce')

if 'ts' in df_conn.columns:
    df_conn['ts'] = pd.to_numeric(df_conn['ts'], errors='coerce')
    df_conn['timestamp'] = pd.to_datetime(df_conn['ts'], unit='s', errors='coerce')

print("✓ Data types converted")

In [ ]:
# ============================================================================
# DISPLAY SAMPLE
# ============================================================================
print(f"\nDataFrame ready: {df_conn.shape}")
print("\nFirst 5 rows:")
display(df_conn.head())

print("\n" + "="*80)
print("✓ CELL 1 COMPLETE - Run Cell 2 next")
print("="*80)

In [ ]:
# ============================================================================
# CELL 2: EXPLORATORY DATA ANALYSIS
# ============================================================================
# Run this cell after Cell 1 completes successfully
# ============================================================================

# Check prerequisite
assert 'df_conn' in dir() and df_conn is not None, \
    "❌ Run Cell 1 first to load df_conn!"

print("="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

In [ ]:
# ============================================================================
# DATASET OVERVIEW
# ============================================================================
print("\n--- DATASET OVERVIEW ---")
print(f"Shape: {df_conn.shape}")
print(f"\nColumns ({len(df_conn.columns)}):")
print(df_conn.columns.tolist())
print(f"\nData types:")
print(df_conn.dtypes)
print(f"\nRandom sample (10 rows):")
display(df_conn.sample(min(10, len(df_conn))))
print("\nStatistical summary:")
display(df_conn.describe(include='all').T)

In [ ]:
# ============================================================================
# MISSING VALUES ANALYSIS
# ============================================================================
print("\n--- MISSING VALUES ANALYSIS ---")

missing_stats = pd.DataFrame({
    'Column': df_conn.columns,
    'Missing_Count': df_conn.isnull().sum(),
    'Missing_Percentage': (df_conn.isnull().sum() / len(df_conn) * 100).round(2),
    'Data_Type': df_conn.dtypes
})
missing_stats = missing_stats[missing_stats['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

if not missing_stats.empty:
    print("\nMissing Values Summary:")
    display(missing_stats)

    # Visualizations
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Heatmap (sample)
    sample_df = df_conn.sample(min(10000, len(df_conn)))
    sns.heatmap(sample_df.isnull(), cbar=True, cmap='viridis',
                ax=axes[0], yticklabels=False)
    axes[0].set_title('Missing Data Heatmap (10k sample)', fontsize=14, weight='bold')

    # Bar chart
    missing_stats.head(15).plot(
        x='Column', y='Missing_Percentage',
        kind='barh', ax=axes[1], color='coral', legend=False
    )
    axes[1].set_title('Missing Data % by Column (Top 15)', fontsize=14, weight='bold')
    axes[1].set_xlabel('Missing Percentage (%)')
    axes[1].set_ylabel('Column')

    plt.tight_layout()
    plt.show()
else:
    print("\n✓ No missing values found")

In [ ]:
# ============================================================================
# NUMERICAL FEATURES DISTRIBUTION
# ============================================================================
print("\n--- NUMERICAL FEATURES DISTRIBUTION ---")

numerical_cols = [c for c in df_conn.select_dtypes(include=np.number).columns
                  if c not in ['ts']]
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols}")

key_numerical = [c for c in ['duration', 'orig_bytes', 'resp_bytes',
                              'orig_pkts', 'resp_pkts', 'orig_ip_bytes', 'resp_ip_bytes']
                 if c in df_conn.columns]

if key_numerical:
    plot_features = key_numerical[:4]
    fig, axes = plt.subplots(len(plot_features), 2,
                             figsize=(16, 4 * len(plot_features)))

    if len(plot_features) == 1:
        axes = axes.reshape(1, -1)

    for idx, col in enumerate(plot_features):
        data = df_conn[col].dropna()

        if len(data) == 0:
            continue

        # Histogram
        axes[idx, 0].hist(data, bins=50, color='skyblue',
                         edgecolor='black', alpha=0.7)
        axes[idx, 0].set_title(f'{col} - Distribution', weight='bold')
        axes[idx, 0].set_xlabel(col)
        axes[idx, 0].set_ylabel('Frequency')

        mean_val = data.mean()
        median_val = data.median()
        axes[idx, 0].axvline(mean_val, color='red', linestyle='--',
                            label=f'Mean: {mean_val:.2f}')
        axes[idx, 0].axvline(median_val, color='green', linestyle='--',
                            label=f'Median: {median_val:.2f}')
        axes[idx, 0].legend()
        axes[idx, 0].set_yscale('log')

        # Boxplot
        axes[idx, 1].boxplot(data, vert=False)
        axes[idx, 1].set_title(f'{col} - Box Plot (Outlier Detection)', weight='bold')
        axes[idx, 1].set_xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# CATEGORICAL FEATURES ANALYSIS
# ============================================================================
print("\n--- CATEGORICAL FEATURES ANALYSIS ---")

categorical_cols = df_conn.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

key_categorical = [c for c in ['proto', 'service', 'conn_state', 'history']
                   if c in df_conn.columns]

for col in key_categorical:
    print(f"\n{col} - Value Counts:")
    value_counts = df_conn[col].value_counts()
    print(value_counts.head(15))

    # Visualization
    fig, ax = plt.subplots(figsize=(12, 5))
    value_counts.head(15).plot(kind='bar', color='teal', ax=ax)
    ax.set_title(f'{col} - Distribution (Top 15)', fontsize=14, weight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# TEMPORAL ANALYSIS
# ============================================================================
print("\n--- TEMPORAL ANALYSIS ---")

if 'timestamp' in df_conn.columns and df_conn['timestamp'].notna().any():
    # Extract temporal features
    df_conn['hour'] = df_conn['timestamp'].dt.hour
    df_conn['day_of_week'] = df_conn['timestamp'].dt.dayofweek
    df_conn['date'] = df_conn['timestamp'].dt.date

    fig, axes = plt.subplots(3, 1, figsize=(16, 12))

    # By hour
    hour_dist = df_conn['hour'].value_counts().sort_index()
    axes[0].plot(hour_dist.index, hour_dist.values, marker='o',
                color='blue', linewidth=2)
    axes[0].fill_between(hour_dist.index, hour_dist.values, alpha=0.3)
    axes[0].set_title('Connection Distribution by Hour of Day',
                     fontsize=14, weight='bold')
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('Number of Connections')
    axes[0].set_xticks(range(24))
    axes[0].grid(True, alpha=0.3)

    # By day of week
    day_dist = df_conn['day_of_week'].value_counts().sort_index()
    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    axes[1].bar(range(7), [day_dist.get(i, 0) for i in range(7)], color='orange')
    axes[1].set_title('Connection Distribution by Day of Week',
                     fontsize=14, weight='bold')
    axes[1].set_xlabel('Day of Week')
    axes[1].set_ylabel('Number of Connections')
    axes[1].set_xticks(range(7))
    axes[1].set_xticklabels(day_names)
    axes[1].grid(True, alpha=0.3, axis='y')

    # Over time
    date_dist = df_conn.groupby('date').size()
    axes[2].plot(range(len(date_dist)), date_dist.values,
                marker='o', color='green', linewidth=2)
    axes[2].set_title('Connection Volume Over Time',
                     fontsize=14, weight='bold')
    axes[2].set_xlabel('Date Index')
    axes[2].set_ylabel('Number of Connections')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\nTemporal range: {df_conn['timestamp'].min()} → {df_conn['timestamp'].max()}")
else:
    print("⚠ No valid timestamp data found. Temporal analysis skipped.")

In [ ]:
# ============================================================================
# CORRELATION ANALYSIS
# ============================================================================
print("\n--- CORRELATION ANALYSIS ---")

numeric_for_corr = [c for c in numerical_cols if c in df_conn.columns]

if len(numeric_for_corr) > 1:
    sample_size = min(50000, len(df_conn))
    df_sample = df_conn[numeric_for_corr].sample(sample_size)
    corr_matrix = df_sample.corr()

    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=1, ax=ax,
                cbar_kws={"shrink": 0.8})
    ax.set_title('Feature Correlation Matrix', fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()

    print("\nHighly Correlated Pairs (|r| > 0.7):")
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            val = corr_matrix.iloc[i, j]
            if abs(val) > 0.7:
                high_corr.append({
                    'Feature_1': corr_matrix.columns[i],
                    'Feature_2': corr_matrix.columns[j],
                    'Correlation': round(val, 4)
                })

    if high_corr:
        high_corr_df = pd.DataFrame(high_corr).sort_values(
            'Correlation', key=abs, ascending=False
        )
        display(high_corr_df)
    else:
        print("  No highly correlated pairs found")

In [ ]:
# ============================================================================
# OUTLIER DETECTION (IQR METHOD)
# ============================================================================
print("\n--- OUTLIER DETECTION (IQR) ---")

outlier_summary = []
for col in key_numerical:
    data = df_conn[col].dropna()
    if len(data) == 0:
        continue

    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data < lower_bound) | (data > upper_bound)]

    outlier_summary.append({
        'Feature': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound,
        'Outlier_Count': len(outliers),
        'Outlier_Percentage': round(len(outliers) / len(data) * 100, 2)
    })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary)
    print("\nOutlier Summary:")
    display(outlier_df)

In [ ]:
# ============================================================================
# NETWORK-SPECIFIC PATTERNS
# ============================================================================
print("\n--- NETWORK-SPECIFIC PATTERN ANALYSIS ---")

# Protocol distribution
if 'proto' in df_conn.columns:
    print(f"\nProtocol Distribution:")
    proto_dist = df_conn['proto'].value_counts()
    print(proto_dist)

    fig, ax = plt.subplots(figsize=(8, 6))
    proto_dist.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=90)
    ax.set_title('Protocol Distribution', fontsize=14, weight='bold')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

# Suspicious connection states
if 'conn_state' in df_conn.columns:
    suspicious_states = ['REJ', 'RSTO', 'RSTOS0', 'RSTR', 'S0', 'S1']
    n_suspicious = df_conn['conn_state'].isin(suspicious_states).sum()
    print(f"\nSuspicious connection states: {n_suspicious:,} "
          f"({n_suspicious / len(df_conn) * 100:.2f}%)")

# Asymmetric flows
if 'orig_bytes' in df_conn.columns and 'resp_bytes' in df_conn.columns:
    df_conn['bytes_ratio'] = np.where(
        df_conn['resp_bytes'] != 0,
        df_conn['orig_bytes'] / df_conn['resp_bytes'],
        0
    )
    df_conn['bytes_ratio'].replace([np.inf, -np.inf], 0, inplace=True)

    n_asymmetric = (
        (df_conn['bytes_ratio'] > 10) | (df_conn['bytes_ratio'] < 0.1)
    ).sum()
    print(f"Highly asymmetric connections: {n_asymmetric:,} "
          f"({n_asymmetric / len(df_conn) * 100:.2f}%)")

print("\n" + "="*80)
print("✓ CELL 2 COMPLETE - Run Cell 3 next")
print("="*80)



In [ ]:
# ============================================================================
# STEP 1: DATA CLEANING
# ============================================================================
print("\n[STEP 1] DATA CLEANING & PREPROCESSING")

df_processed = df_conn.copy()
print(f"  Starting: {df_processed.shape}")

# Drop identifiers and always-empty columns
drop_cols = ['uid', 'local_orig', 'local_resp', 'tunnel_parents',
             'timestamp', 'date']
df_processed.drop(columns=[c for c in drop_cols if c in df_processed.columns],
                  inplace=True)

# Drop columns with >50% missing
missing_pct = df_processed.isnull().mean()
high_missing = missing_pct[missing_pct > 0.5].index.tolist()
df_processed.drop(columns=high_missing, inplace=True)
print(f"  Dropped high-missing: {high_missing}")

# Impute numeric → median
num_cols = [c for c in df_processed.select_dtypes(include=np.number).columns
            if c not in ['ts']]
df_processed[num_cols] = df_processed[num_cols].fillna(
    df_processed[num_cols].median()
)

# Impute categorical → mode
cat_cols = df_processed.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    if len(df_processed[c].mode()) > 0:
        df_processed[c] = df_processed[c].fillna(df_processed[c].mode()[0])

# Remove duplicates
before_dup = len(df_processed)
df_processed.drop_duplicates(inplace=True)
print(f"  Removed {before_dup - len(df_processed):,} duplicates")

# Handle infinities
df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
num_cols = [c for c in df_processed.select_dtypes(include=np.number).columns
            if c not in ['ts']]
df_processed[num_cols] = df_processed[num_cols].fillna(
    df_processed[num_cols].median()
)

# Encode categorical
for col in df_processed.select_dtypes(include='object').columns:
    df_processed[col] = LabelEncoder().fit_transform(df_processed[col].astype(str))

print(f"  ✓ Cleaned: {df_processed.shape}, missing: {df_processed.isnull().sum().sum()}")

# STEP 2: FEATURE ENGINEERING
# ============================================================================
print("\n[STEP 2] FEATURE ENGINEERING")

# Temporal features
if 'hour' in df_processed.columns:
    df_processed['is_business_hours'] = df_processed['hour'].between(9, 17).astype(int)
    df_processed['is_night'] = ((df_processed['hour'] >= 22) |
                                (df_processed['hour'] <= 6)).astype(int)
    print("  ✓ Temporal features created")

if 'day_of_week' in df_processed.columns:
    df_processed['is_weekend'] = (df_processed['day_of_week'] >= 5).astype(int)

# Byte features
if 'orig_bytes' in df_processed.columns and 'resp_bytes' in df_processed.columns:
    df_processed['bytes_ratio'] = (
        df_processed['orig_bytes'] / (df_processed['resp_bytes'] + 1)
    )
    df_processed['total_bytes'] = (
        df_processed['orig_bytes'] + df_processed['resp_bytes']
    )

    if 'orig_pkts' in df_processed.columns:
        df_processed['avg_bytes_per_orig_pkt'] = np.where(
            df_processed['orig_pkts'] != 0,
            df_processed['orig_bytes'] / df_processed['orig_pkts'], 0
        )
    if 'resp_pkts' in df_processed.columns:
        df_processed['avg_bytes_per_resp_pkt'] = np.where(
            df_processed['resp_pkts'] != 0,
            df_processed['resp_bytes'] / df_processed['resp_pkts'], 0
        )
    print("  ✓ Byte features created")

# Packet features
if 'orig_pkts' in df_processed.columns and 'resp_pkts' in df_processed.columns:
    df_processed['pkt_ratio'] = (
        df_processed['orig_pkts'] / (df_processed['resp_pkts'] + 1)
    )
    df_processed['total_pkts'] = (
        df_processed['orig_pkts'] + df_processed['resp_pkts']
    )
    print("  ✓ Packet features created")

# Duration features
if 'duration' in df_processed.columns:
    df_processed['log_duration'] = np.log1p(df_processed['duration'])
    df_processed['is_very_short_duration'] = (
        df_processed['duration'] < 0.1
    ).astype(int)

    if 'total_bytes' in df_processed.columns:
        df_processed['bytes_per_second'] = np.where(
            df_processed['duration'] != 0,
            df_processed['total_bytes'] / df_processed['duration'], 0
        )
    print("  ✓ Duration features created")

# Flag features
if 'total_bytes' in df_processed.columns:
    df_processed['is_zero_bytes'] = (df_processed['total_bytes'] == 0).astype(int)

if 'total_pkts' in df_processed.columns:
    df_processed['is_single_packet'] = (df_processed['total_pkts'] <= 1).astype(int)

# Clean infinities
df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
df_processed.fillna(df_processed.median(numeric_only=True), inplace=True)

print(f"  ✓ Final shape: {df_processed.shape}")


In [ ]:
# ============================================================================
# STEP 3: FEATURE SELECTION
# ============================================================================
print("\n[STEP 3] FEATURE SELECTION")

X = df_processed.select_dtypes(include=np.number).copy()
drop_ids = ['ts', 'id.orig_p', 'id.resp_p']
X.drop(columns=[c for c in drop_ids if c in X.columns], inplace=True)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)
print(f"  Starting features: {X.shape[1]}")

# Variance threshold
sel_var = VarianceThreshold(threshold=0.01)
X_var_arr = sel_var.fit_transform(X)
selected_features_var = X.columns[sel_var.get_support()].tolist()
X_var = pd.DataFrame(X_var_arr, columns=selected_features_var)
print(f"  After variance filter: {X_var.shape[1]}")

# Correlation filter
corr_matrix_sel = X_var.corr().abs()
upper = corr_matrix_sel.where(
    np.triu(np.ones(corr_matrix_sel.shape), k=1).astype(bool)
)
high_corr_drop = [c for c in upper.columns if any(upper[c] > 0.90)]
X_corr = X_var.drop(columns=high_corr_drop)
print(f"  After correlation filter: {X_corr.shape[1]}")

# Feature importance
print("  Computing feature importance...")
iso_temp = IsolationForest(n_estimators=100, contamination=0.1,
                           random_state=42, n_jobs=-1, verbose=0)
iso_temp.fit(X_corr)
anomaly_scores_temp = iso_temp.decision_function(X_corr)

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42,
                                 n_jobs=-1, verbose=0)
y_temp = (anomaly_scores_temp < np.percentile(anomaly_scores_temp, 10)).astype(int)
rf_temp.fit(X_corr, y_temp)

feature_importance = pd.DataFrame({
    'feature': X_corr.columns,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("  Top 20 features:")
display(feature_importance.head(20))

fig, ax = plt.subplots(figsize=(12, 8))
feature_importance.head(20).plot(x='feature', y='importance', kind='barh',
                                ax=ax, color='steelblue', legend=False)
ax.set_title('Top 20 Feature Importances', fontsize=14, weight='bold')
ax.set_xlabel('Importance')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Select top features
top_n = min(30, len(feature_importance))
selected_features_final = feature_importance.head(top_n)['feature'].tolist()
X_final = X_corr[selected_features_final].copy()
print(f"  ✓ Final selected: {X_final.shape[1]} features")

In [ ]:
#============================================================================
# STEP 4: SCALING
# ============================================================================
print("\n[STEP 4] SCALING")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)
X_train_scaled, X_test_scaled = train_test_split(
    X_scaled, test_size=0.2, random_state=42
)
print(f"  Train: {X_train_scaled.shape}  |  Test: {X_test_scaled.shape}")


In [ ]:
# ============================================================================
# STEP 5: ANOMALY DETECTION
# ============================================================================
print("\n[STEP 5] ANOMALY DETECTION")

models = {}
results = {}
scores_dict = {}

# Isolation Forest
print("  [5.1] Isolation Forest")
iso_forest = IsolationForest(n_estimators=200, contamination=0.1,
                             random_state=42, n_jobs=-1, verbose=0)
iso_forest.fit(X_train_scaled)
y_pred_iso = iso_forest.predict(X_test_scaled)
y_pred_iso_binary = (y_pred_iso == -1).astype(int)
scores_iso = iso_forest.decision_function(X_test_scaled)

models["IsolationForest"] = iso_forest
results["IsolationForest"] = y_pred_iso_binary
scores_dict["IsolationForest"] = scores_iso
print(f"    Anomalies: {y_pred_iso_binary.sum():,} / {len(y_pred_iso_binary):,} "
      f"({y_pred_iso_binary.mean()*100:.2f}%)")

# One-Class SVM
print("  [5.2] One-Class SVM")
n_svm = min(50_000, len(X_train_scaled))
oc_svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
oc_svm.fit(X_train_scaled[:n_svm])
y_pred_svm = oc_svm.predict(X_test_scaled)
y_pred_svm_binary = (y_pred_svm == -1).astype(int)
scores_svm = oc_svm.decision_function(X_test_scaled)

models["OneClassSVM"] = oc_svm
results["OneClassSVM"] = y_pred_svm_binary
scores_dict["OneClassSVM"] = scores_svm
print(f"    Anomalies: {y_pred_svm_binary.sum():,} / {len(y_pred_svm_binary):,} "
      f"({y_pred_svm_binary.mean()*100:.2f}%)")

# Local Outlier Factor
print("  [5.3] Local Outlier Factor")
n_lof = min(50_000, len(X_train_scaled))
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1,
                         novelty=True, n_jobs=-1)
lof.fit(X_train_scaled[:n_lof])
y_pred_lof = lof.predict(X_test_scaled)
y_pred_lof_binary = (y_pred_lof == -1).astype(int)
scores_lof = lof.decision_function(X_test_scaled)

models["LOF"] = lof
results["LOF"] = y_pred_lof_binary
scores_dict["LOF"] = scores_lof
print(f"    Anomalies: {y_pred_lof_binary.sum():,} / {len(y_pred_lof_binary):,} "
      f"({y_pred_lof_binary.mean()*100:.2f}%)")

# Autoencoder (optional)
print("  [5.4] Autoencoder (optional)")
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers as tf_layers

    autoencoder = keras.Sequential([
        tf_layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        tf_layers.Dense(32, activation='relu'),
        tf_layers.Dense(5, activation='relu'),
        tf_layers.Dense(32, activation='relu'),
        tf_layers.Dense(64, activation='relu'),
        tf_layers.Dense(X_train_scaled.shape[1], activation='linear')
    ])
    autoencoder.compile(optimizer='adam', loss='mse')
    autoencoder.fit(X_train_scaled, X_train_scaled, epochs=20,
                   batch_size=256, verbose=0, validation_split=0.1)

    train_recon = autoencoder.predict(X_train_scaled, verbose=0)
    train_err = np.mean(np.square(X_train_scaled - train_recon), axis=1)
    threshold_ae = np.percentile(train_err, 95)

    test_recon = autoencoder.predict(X_test_scaled, verbose=0)
    scores_ae = np.mean(np.square(X_test_scaled - test_recon), axis=1)
    y_pred_ae = (scores_ae > threshold_ae).astype(int)

    models["Autoencoder"] = autoencoder
    results["Autoencoder"] = y_pred_ae
    scores_dict["Autoencoder"] = -scores_ae
    print(f"    Anomalies: {y_pred_ae.sum():,} / {len(y_pred_ae):,} "
          f"({y_pred_ae.mean()*100:.2f}%)")
except:
    print("    Skipped (TensorFlow not available)")


In [ ]:
# ============================================================================
# STEP 6: RESULTS & VISUALIZATIONS
# ============================================================================
print("\n[STEP 6] RESULTS")

# Model comparison table
summary_rows = []
for name, preds in results.items():
    summary_rows.append([name, f"{int(preds.sum()):,}",
                        f"{preds.mean()*100:.2f}%"])

results_df = pd.DataFrame(summary_rows,
                         columns=["Model", "Anomalies", "Percentage"])
print("\n" + results_df.to_string(index=False))

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Network Anomaly Detection Results", fontsize=16, weight='bold')

# PCA scatter
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_test_scaled)
axes[0,0].scatter(X_pca[y_pred_iso_binary==0, 0],
                 X_pca[y_pred_iso_binary==0, 1],
                 s=5, alpha=0.3, color='steelblue', label='Normal')
axes[0,0].scatter(X_pca[y_pred_iso_binary==1, 0],
                 X_pca[y_pred_iso_binary==1, 1],
                 s=20, alpha=0.8, color='red', label='Anomaly')
axes[0,0].set_title("PCA Projection — Isolation Forest", weight='bold')
axes[0,0].legend(markerscale=3)

# Score distribution
axes[0,1].hist(scores_iso, bins=50, color='steelblue',
              edgecolor='black', alpha=0.7)
axes[0,1].axvline(0, color='red', linestyle='--', label='Threshold')
axes[0,1].set_title("Score Distribution", weight='bold')
axes[0,1].set_xlabel("Anomaly Score")
axes[0,1].legend()

# Model comparison
model_names = list(results.keys())
counts = [int(results[m].sum()) for m in model_names]
colors = ['steelblue','coral','mediumseagreen','gold'][:len(counts)]
bars = axes[1,0].bar(model_names, counts, color=colors)
axes[1,0].set_title("Anomalies by Model", weight='bold')
axes[1,0].set_ylabel("Count")
axes[1,0].tick_params(axis='x', rotation=15)
for bar, cnt in zip(bars, counts):
    axes[1,0].text(bar.get_x() + bar.get_width()/2,
                   bar.get_height() + max(counts)*0.01,
                   f"{cnt:,}", ha='center', fontsize=9)

# Sorted scores
axes[1,1].plot(np.sort(scores_iso), color='steelblue', linewidth=1.5)
axes[1,1].axhline(0, linestyle='--', color='red', label='Threshold')
axes[1,1].set_title("Sorted Anomaly Scores", weight='bold')
axes[1,1].set_xlabel("Connection index (sorted)")
axes[1,1].set_ylabel("Score")
axes[1,1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)
print(f"  Records analysed:    {df_processed.shape[0]:,}")
print(f"  Features selected:   {X_final.shape[1]}")
print(f"  Test set size:       {X_test_scaled.shape[0]:,}")
print()
for name, preds in results.items():
    print(f"  {name:<20}: {int(preds.sum()):>7,} anomalies "
          f"({preds.mean()*100:.2f}%)")
print("\n✓ All cells complete - pipeline finished successfully!")


In [ ]:
# ============================================================
# EXPORT FOR KIBANA
# ============================================================
import pandas as pd
import numpy as np

# Model comparison
comparison_df = pd.DataFrame({
    'model': ['Isolation Forest', 'One Class SVM', 'LOF'],
    'total_records': [len(y_pred_iso_binary), len(y_pred_svm_binary), len(y_pred_lof_binary)],
    'anomalies_detected': [int(y_pred_iso_binary.sum()), int(y_pred_svm_binary.sum()), int(y_pred_lof_binary.sum())],
    'normal_records': [int((y_pred_iso_binary==0).sum()), int((y_pred_svm_binary==0).sum()), int((y_pred_lof_binary==0).sum())],
    'anomaly_percentage': [round(y_pred_iso_binary.mean()*100,2), round(y_pred_svm_binary.mean()*100,2), round(y_pred_lof_binary.mean()*100,2)]
})

# Full results
results_export = pd.DataFrame({
    'record_index': range(len(y_pred_iso_binary)),
    'isolation_forest_label': ['Anomaly' if x==1 else 'Normal' for x in y_pred_iso_binary],
    'isolation_forest_score': scores_iso,
    'ocsvm_label': ['Anomaly' if x==1 else 'Normal' for x in y_pred_svm_binary],
    'ocsvm_score': scores_svm,
    'lof_label': ['Anomaly' if x==1 else 'Normal' for x in y_pred_lof_binary],
    'lof_score': scores_lof
})

# Save and download
comparison_df.to_csv('model_comparison.csv', index=False)
results_export.to_csv('ml_full_results.csv', index=False)

from google.colab import files
files.download('model_comparison.csv')
files.download('ml_full_results.csv')

print("✅ Files downloaded!")
print("\n=== MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))
